In [2]:
import pandas as pd

langs = ['Arabic', 'Chinese', 'French', 'Japanese', 'Russian']
for lang in langs:
    print(lang)
    df = pd.read_csv(f"annotation/{lang}.csv")
    df_ref = pd.read_csv(f"dataset/{lang}.csv")
    for idx, row in df.iterrows():
        for i in range(1, 6):
            if not df_ref.loc[(df_ref['en_word1'] == row['Input.en_word1']) & (df_ref['en_word5'] == row['Input.en_word5'])].empty:
                first_row = df_ref.loc[(df_ref['en_word1'] == row['Input.en_word1']) & (df_ref['en_word5'] == row['Input.en_word5'])].iloc[0]
                df.loc[idx, f'source{i}'] = first_row[f'source{i}']

    # Convert rows into multiple rows based on "i"
    rows = []
    for _, row in df.iterrows():
        for i in range(1, 6):  # For each "i" from 1 to 5
            new_row = {
                "WorkerId": row["WorkerId"],
                "en_word": row[f"Input.en_word{i}"],
                "anno1_word": row[f"Input.anno1_word{i}"],
                "anno2_word": row[f"Input.anno2_word{i}"],
                "Answer.A": row[f"Answer.A.{i}"],
                "Answer.B": row[f"Answer.B.{i}"],
                "Answer.C": row[f"Answer.C.{i}"],
                "Answer.D": row[f"Answer.D.{i}"],
                "source": row[f"source{i}"],
            }
            rows.append(new_row)

    # Create the new DataFrame
    df = pd.DataFrame(rows)
    df.to_csv(f"annotation/{lang}_als.csv")

Arabic
Chinese
French
Japanese
Russian


In [ ]:
sources = ['6060', 'threshold']

new_dfs = {}
for lang in langs:
    df = pd.read_csv(f"annotation/{lang}_als.csv")
    
    new_df = pd.DataFrame(columns=['A', 'B', 'C', 'D'])
    for source in sources:
        new_df.loc[source] = [None, None, None, None]
        count_total = df.loc[df['source'] == source].shape[0]
        for idx, choice in enumerate(['A', 'B', 'C', 'D']):
            count_true = df.loc[df['source'] == source][f'Answer.{choice}'].sum()
            ratio_true = count_true/count_total
            res_str = f"{count_true}, {ratio_true:.2%}"
            new_df.loc[source][idx] = res_str
    
    # print(new_df)
    # print(new_df.to_latex())
    new_dfs[lang] = new_df.to_latex()

In [5]:
for lang in new_dfs:
    print(lang)
    print(new_dfs[lang])

Arabic
\begin{tabular}{lllll}
\toprule
 & A & B & C & D \\
\midrule
6060 & 376, 46.42% & 238, 29.38% & 143, 17.65% & 46, 5.68% \\
threshold & 739, 45.76% & 461, 28.54% & 329, 20.37% & 72, 4.46% \\
\bottomrule
\end{tabular}

Chinese
\begin{tabular}{lllll}
\toprule
 & A & B & C & D \\
\midrule
6060 & 197, 37.17% & 228, 43.02% & 85, 16.04% & 17, 3.21% \\
threshold & 468, 50.59% & 266, 28.76% & 163, 17.62% & 25, 2.70% \\
\bottomrule
\end{tabular}

French
\begin{tabular}{lllll}
\toprule
 & A & B & C & D \\
\midrule
6060 & 152, 39.48% & 168, 43.64% & 53, 13.77% & 10, 2.60% \\
threshold & 438, 48.67% & 274, 30.44% & 170, 18.89% & 17, 1.89% \\
\bottomrule
\end{tabular}

Japanese
\begin{tabular}{lllll}
\toprule
 & A & B & C & D \\
\midrule
6060 & 295, 57.28% & 162, 31.46% & 36, 6.99% & 21, 4.08% \\
threshold & 587, 56.99% & 251, 24.37% & 159, 15.44% & 27, 2.62% \\
\bottomrule
\end{tabular}

Russian
\begin{tabular}{lllll}
\toprule
 & A & B & C & D \\
\midrule
6060 & 172, 39.09% & 198, 45.00% & 3

## inter annotator agreement

In [35]:
import pandas as pd
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa

sources = ['6060', 'threshold']
langs = ['Arabic', 'Chinese', 'French', 'Japanese', 'Russian']
for lang in langs:
    print(lang)
    df = pd.read_csv(f"annotation/{lang}_als.csv")
    
    for source in sources:
        print(source)
        data = df.loc[df['source'] == source]

        def map_choice(row):
            if row["Answer.A"]:
                return 0  # A
            elif row["Answer.B"]:
                return 1  # B
            elif row["Answer.C"]:
                return 2  # C
            elif row["Answer.D"]:
                return 3  # D
            return -1  # Shouldn't happen if data is valid

        data["Choice"] = data.apply(map_choice, axis=1)

        # Step 2: Create a matrix for Fleiss' Kappa
        # Count how many annotators selected each option for each question
        questions = data[["en_word", "anno1_word", "anno2_word"]].drop_duplicates()
        kappa_matrix = []

        for _, question in questions.iterrows():
            question_data = data[((data["en_word"] == question["en_word"]) & 
                         (data["anno1_word"] == question["anno1_word"])) & (data["anno2_word"] == question["anno2_word"])]
            counts = question_data["Choice"].value_counts().reindex(range(4), fill_value=0)
            
            # Ensure consistent annotator counts
            if counts.sum() != 5:
                # print(f"Skipping question {question} due to inconsistent annotations.")
                continue

            kappa_matrix.append(counts.tolist())

        # Convert to NumPy array
        kappa_matrix = np.array(kappa_matrix)

        # Step 3: Calculate Fleiss' Kappa
        if len(kappa_matrix) > 0:
            fleiss_score = fleiss_kappa(kappa_matrix, method='fleiss')
            print(f"Fleiss' Kappa: {fleiss_score:.2f}")
        else:
            print("No valid questions to calculate Fleiss' Kappa.")

Arabic
6060


/tmp/ipykernel_2652924/1723126277.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)


Fleiss' Kappa: 0.22
threshold


/tmp/ipykernel_2652924/1723126277.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)


Fleiss' Kappa: 0.30
Chinese
6060
Fleiss' Kappa: 0.41
threshold


...

Fleiss' Kappa: 0.37
Japanese
6060
Fleiss' Kappa: 0.39
threshold


/tmp/ipykernel_2652924/1723126277.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)
/tmp/ipykernel_2652924/1723126277.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)


Fleiss' Kappa: 0.21
Russian
6060
Fleiss' Kappa: 0.41
threshold


/tmp/ipykernel_2652924/1723126277.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)
/tmp/ipykernel_2652924/1723126277.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)


Fleiss' Kappa: 0.39


In [61]:
import pandas as pd
import numpy as np
import krippendorff
from statsmodels.stats.inter_rater import fleiss_kappa

sources = ['6060', 'threshold']
langs = ['Arabic', 'Chinese', 'French', 'Japanese', 'Russian']
for lang in langs:
    print(lang)
    df = pd.read_csv(f"annotation/{lang}_als.csv")
    
    for source in sources:
        print(source)
        data = df.loc[df['source'] == source]

        def map_choice(row):
            if row["Answer.A"]:
                return 0  # A
            elif row["Answer.B"]:
                return 1  # B
            elif row["Answer.C"]:
                return 2  # C
            elif row["Answer.D"]:
                return 3  # D
            return 0  # Shouldn't happen if data is valid

        data["Choice"] = data.apply(map_choice, axis=1)

        # Step 2: Create a matrix for Fleiss' Kappa
        # Count how many annotators selected each option for each question
        questions = data[["en_word", "anno1_word", "anno2_word"]].drop_duplicates()
        reliability_matrix = []

        for _, question in questions.iterrows():
            question_data = data[((data["en_word"] == question["en_word"]) & 
                         (data["anno1_word"] == question["anno1_word"])) & (data["anno2_word"] == question["anno2_word"])]
            if question_data.shape[0] != 5:
                continue
            reliability_matrix.append(question_data['Choice'].values.tolist())
    
    
        reliability_matrix = np.array(reliability_matrix).T  # Transpose for annotator × questions

        # Step 3: Compute Krippendorff's Alpha
        alpha = krippendorff.alpha(reliability_data=reliability_matrix, level_of_measurement="nominal")
        print(alpha)

Arabic
6060
0.21394600675961528
threshold


/tmp/ipykernel_2652924/3565351378.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)
/tmp/ipykernel_2652924/3565351378.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)


0.28846782017118366
Chinese
6060
0.3978098367162436
threshold


/tmp/ipykernel_2652924/3565351378.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)
/tmp/ipykernel_2652924/3565351378.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)


0.22142678516107595
French
6060
0.5085675119945168
threshold


...

0.37600375022787835
Japanese
6060
0.3839673875776568
threshold


/tmp/ipykernel_2652924/3565351378.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)
/tmp/ipykernel_2652924/3565351378.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)


0.20167349501815757
Russian
6060
0.38868613138686137
threshold


/tmp/ipykernel_2652924/3565351378.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)
/tmp/ipykernel_2652924/3565351378.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Choice"] = data.apply(map_choice, axis=1)


0.3874720175529628
